In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("arashnic/fitbit")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\Karolcia\.cache\kagglehub\datasets\arashnic\fitbit\versions\2


In [3]:
import pandas as pd
import numpy as np
import pyspark as psp
import os
import ast
import seaborn as sns
import matplotlib.pyplot as plt


In [4]:

dataset_path_03_01 = os.path.join(path,'mturkfitbit_export_3.12.16-4.11.16','Fitabase Data 3.12.16-4.11.16','dailyActivity_merged.csv')
df_act_marged3 = pd.read_csv(dataset_path_03_01)  ##, delimiter="\t")
df3 = pd.DataFrame(df_act_marged3)


dataset_path_04_01 = os.path.join(path,'mturkfitbit_export_4.12.16-5.12.16','Fitabase Data 4.12.16-5.12.16','dailyActivity_merged.csv')
df_act_marged4 = pd.read_csv(dataset_path_04_01)  ##, delimiter="\t")
df4 = pd.DataFrame(df_act_marged4)

df = pd.concat([df3, df4], ignore_index=True)



In [5]:
df.head(5)

,Id,ActivityDate,TotalSteps,TotalDistance,TrackerDistance,LoggedActivitiesDistance,VeryActiveDistance,ModeratelyActiveDistance,LightActiveDistance,SedentaryActiveDistance,VeryActiveMinutes,FairlyActiveMinutes,LightlyActiveMinutes,SedentaryMinutes,Calories
0,1503960366,3/25/2016,11004,7.11,7.11,0.0,2.57,0.46,4.07,0.0,33,12,205,804,1819
1,1503960366,3/26/2016,17609,11.55,11.55,0.0,6.92,0.73,3.91,0.0,89,17,274,588,2154
2,1503960366,3/27/2016,12736,8.53,8.53,0.0,4.66,0.16,3.71,0.0,56,5,268,605,1944
3,1503960366,3/28/2016,13231,8.93,8.93,0.0,3.19,0.79,4.95,0.0,39,20,224,1080,1932
4,1503960366,3/29/2016,12041,7.85,7.85,0.0,2.16,1.09,4.61,0.0,28,28,243,763,1886


In [7]:

dataset_path_03_01_wght = os.path.join(path,'mturkfitbit_export_3.12.16-4.11.16','Fitabase Data 3.12.16-4.11.16','weightLogInfo_merged.csv')
df_act_marged3_wght  = pd.read_csv(dataset_path_03_01_wght)  ##, delimiter="\t")
df3_wght = pd.DataFrame(df_act_marged3_wght)


dataset_path_04_01_wght = os.path.join(path,'mturkfitbit_export_4.12.16-5.12.16','Fitabase Data 4.12.16-5.12.16','weightLogInfo_merged.csv')
df_act_marged4_wght  = pd.read_csv(dataset_path_04_01_wght)  ##, delimiter="\t")
df4_wght = pd.DataFrame(df_act_marged4_wght)

df_wght = pd.concat([df3_wght, df4_wght], ignore_index=True)


In [64]:
df_wght.head(15)
df_wght['join'] = df_wght['Id'].astype(str) + df_wght['Date'].astype(str)

df_wght[df_wght['Id'].astype(str) == '8877689391'].sort_values(by='Date').head(15)

,Id,Date,WeightKg,WeightPounds,Fat,BMI,IsManualReport,LogId,join
24,8877689391,4/1/2016 6:49:40 AM,85.500000,188.495234,NaN,25.610001,False,1459493380000,88776893914/1/2016 6:49:40 AM
31,8877689391,4/11/2016 6:58:09 AM,86.099998,189.818004,NaN,25.790001,False,1460357889000,88776893914/11/2016 6:58:09 AM
32,8877689391,4/12/2016 6:47:11 AM,85.800003,189.156628,NaN,25.680000,False,1460443631000,88776893914/12/2016 6:47:11 AM
76,8877689391,4/12/2016 6:47:11 AM,85.800003,189.156628,NaN,25.680000,False,1460443631000,88776893914/12/2016 6:47:11 AM
77,8877689391,4/13/2016 6:55:00 AM,84.900002,187.172464,NaN,25.410000,False,1460530500000,88776893914/13/2016 6:55:00 AM
78,8877689391,4/14/2016 6:48:43 AM,84.500000,186.290612,NaN,25.309999,False,1460616523000,88776893914/14/2016 6:48:43 AM
79,8877689391,4/16/2016 1:39:25 PM,85.500000,188.495234,NaN,25.590000,False,1460813965000,88776893914/16/2016 1:39:25 PM
80,8877689391,4/18/2016 6:51:14 AM,85.800003,189.156628,NaN,25.680000,False,1460962274000,88776893914/18/2016 6:51:14 AM
81,8877689391,4/19/2016 6:39:31 AM,85.300003,188.054316,NaN,25.530001,False,1461047971000,88776893914/19/2016 6:39:31 AM
82,8877689391,4/20/2016 6:44:54 AM,84.900002,187.172464,NaN,25.410000,False,1461134694000,88776893914/20/2016 6:44:54 AM


In [59]:
df_wght_group = df_wght.groupby(['Id'])['Date'].agg(['min','max', 'count']).reset_index()

df_wght_group['join_min'] = df_wght_group['Id'].astype(str) + df_wght_group['min'].astype(str)
df_wght_group['join_max'] = df_wght_group['Id'].astype(str) + df_wght_group['max'].astype(str)
df_wght_join = df_wght.drop(columns=['Id', 'Date', 'WeightPounds', 'IsManualReport', 'LogId'])
df_wght_all = pd.merge(df_wght_group, df_wght_join, left_on=['join_min'], right_on=['join'], how='left', suffixes=('', '_min'))
df_wght_all = pd.merge(df_wght_all, df_wght_join, left_on=['join_max'], right_on=['join'], how='left', suffixes=('', '_max'))

df_wght_all = df_wght_all.drop(columns=['join_max', 'join_min', 'join'])
df_wght_all['WeightKg_diff'] = df_wght_all['WeightKg_max'] - df_wght_all['WeightKg'] 
df_wght_all['BMI_diff'] = df_wght_all['BMI_max'] - df_wght_all['BMI'] 
df_wght_all['Hight'] = np.sqrt(df_wght_all['WeightKg'] / df_wght_all['BMI'])
df_wght_all.head(15)

,Id,min,max,count,WeightKg,Fat,BMI,WeightKg_max,Fat_max,BMI_max,WeightKg_diff,BMI_diff,Hight
0,1503960366,4/5/2016 11:59:59 PM,5/3/2016 11:59:59 PM,3,53.299999,22.0,22.969999,52.599998,NaN,22.650000,-0.700001,-0.320000,1.523292
1,1927972279,4/10/2016 6:33:26 PM,4/13/2016 1:08:52 AM,2,129.600006,NaN,46.169998,133.500000,NaN,47.540001,3.899994,1.370003,1.675416
2,2347167796,4/3/2016 11:59:59 PM,4/3/2016 11:59:59 PM,1,63.400002,10.0,24.770000,63.400002,10.0,24.770000,0.000000,0.000000,1.599859
3,2873212765,4/21/2016 11:59:59 PM,5/12/2016 11:59:59 PM,4,56.700001,NaN,21.450001,57.299999,NaN,21.690001,0.599998,0.240000,1.625840
4,2891001357,4/5/2016 11:59:59 PM,4/5/2016 11:59:59 PM,1,88.400002,NaN,25.030001,88.400002,NaN,25.030001,0.000000,0.000000,1.879298
5,4319703577,4/17/2016 11:59:59 PM,5/4/2016 11:59:59 PM,2,72.400002,25.0,27.450001,72.300003,NaN,27.379999,-0.099998,-0.070002,1.624045
6,4445114986,3/30/2016 11:59:59 PM,3/30/2016 11:59:59 PM,1,92.400002,NaN,35.009998,92.400002,NaN,35.009998,0.000000,0.000000,1.624576
7,4558609924,4/18/2016 11:59:59 PM,5/9/2016 11:59:59 PM,6,69.699997,NaN,27.250000,69.099998,NaN,27.000000,-0.599998,-0.250000,1.599312
8,4702921684,4/4/2016 11:59:59 PM,4/4/2016 11:59:59 PM,1,99.699997,NaN,26.110001,99.699997,NaN,26.110001,0.000000,0.000000,1.954088
9,5577150313,4/17/2016 9:17:55 AM,4/17/2016 9:17:55 AM,1,90.699997,NaN,28.000000,90.699997,NaN,28.000000,0.000000,0.000000,1.799802
